# E-Commerce Sales & Profitability Analysis

## Introduction

I wanted to understand how this e-commerce business is performing and, more importantly, where the money is coming from and where it is being lost.

This analysis looks at sales, profitability, products, regions, countries, customer segments, discounts, and payment methods. The goal is to find a few useful patterns that could help a business make better decisions.

### Questions I wanted to answer

- How have sales changed over time?
- Which products and categories generate the most sales?
- How common are loss-making orders?
- Which products are responsible for the losses?
- Which regions and countries perform best?
- Which customer segment is most valuable?
- Are discounts associated with lower profitability?
- How do payment methods compare?

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the data

In [ ]:
df = pd.read_csv("../data/global_ecommerce_sales.csv")

df.head()

## 3. Initial data check

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
duplicate_orders = df["Order_ID"].duplicated().sum()
print("Duplicate Order_IDs:", duplicate_orders)

### Initial observations

The dataset contains 2,000 orders and 15 columns. I also checked for missing values and duplicate order IDs before starting the analysis.

Since each `Order_ID` represents an order and the IDs are unique, I can use the order ID count as an order count in later summaries.

## 4. Data preparation

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Day_Name"] = df["Order_Date"].dt.day_name()

df.head()

In [ ]:
# Check that Total_Sales agrees with the basic sales calculation:
# quantity × unit price × (1 - discount)

df["calculated_sales"] = (
    df["Quantity"]
    * df["Unit_Price"]
    * (1 - df["Discount_Percent"] / 100)
)

sales_difference = (
    df["Total_Sales"] - df["calculated_sales"]
).abs()

print("Largest sales calculation difference:", sales_difference.max())

In [ ]:
# The calculated column was only used as a validation check.
df.drop(columns="calculated_sales", inplace=True)

## 5. Sales analysis

I started with the simplest question: how much is the business selling, and how does that change over time?

In [ ]:
total_sales = df["Total_Sales"].sum()
total_orders = df["Order_ID"].nunique()

print(f"Total sales: ${total_sales:,.2f}")
print(f"Total orders: {total_orders:,}")
print(f"Average order value: ${total_sales / total_orders:,.2f}")

### Sales by year

In [ ]:
yearly_sales = (
    df.groupby("Year")["Total_Sales"]
    .sum()
    .sort_index()
)

yearly_sales

In [ ]:
yearly_growth = yearly_sales.pct_change() * 100
yearly_growth

In [ ]:
yearly_sales.plot(kind="bar", figsize=(9, 5))
plt.title("Sales by Year")
plt.xlabel("Year")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

The yearly results show that sales were relatively stable across the three years. There was a decline from 2023 to 2024, followed by a recovery in 2025.

Based on the data, sales were:
- 2023: $164,443.45
- 2024: $155,150.73
- 2025: $164,965.16

So 2025 recovered from the previous year's decline and slightly exceeded 2023 sales.

### Monthly sales

In [ ]:
monthly_sales = (
    df.groupby(df["Order_Date"].dt.to_period("M"))["Total_Sales"]
    .sum()
)

monthly_sales

In [ ]:
monthly_sales.plot(kind="line", marker="o", figsize=(11, 5))
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
best_month = monthly_sales.idxmax()
worst_month = monthly_sales.idxmin()

print("Best month:", best_month, f"(${monthly_sales.max():,.2f})")
print("Lowest month:", worst_month, f"(${monthly_sales.min():,.2f})")

### A closer look at February and June 2025

February and June stood out as useful months to compare. June had more orders and more units sold, and its average order value was also much higher.

- February: 40 orders, 114 units, AOV ≈ $183.43
- June: 64 orders, 209 units, AOV ≈ $282.31

The category mix also changed noticeably between the two months, with Technology leading February sales and Furniture becoming much stronger in June.

### Sales by product category

In [ ]:
category_sales = (
    df.groupby("Product_Category")["Total_Sales"]
    .sum()
    .sort_values(ascending=False)
)

category_sales

In [ ]:
category_sales.plot(kind="bar", figsize=(9, 5))
plt.title("Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Top 10 products by sales

In [ ]:
top_products_sales = (
    df.groupby("Product_Name")["Total_Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products_sales

The strongest individual products by sales were led by Standing Desk Converter, Ergonomic Office Chair, Corner L-Shaped Desk, and Mesh Back Task Chair.

The top four were all Furniture products, which is consistent with the strong Furniture performance seen in the June comparison.

## 6. Profitability analysis

In [ ]:
total_profit = df["Profit"].sum()
average_profit = df["Profit"].mean()
profit_margin = total_profit / total_sales * 100

profitable_orders = (df["Profit"] > 0).sum()
loss_orders_count = (df["Profit"] < 0).sum()
zero_profit_orders = (df["Profit"] == 0).sum()

print(f"Total profit: ${total_profit:,.2f}")
print(f"Average profit per order: ${average_profit:,.2f}")
print(f"Overall profit margin: {profit_margin:.2f}%")
print()
print("Profitable orders:", profitable_orders)
print("Loss-making orders:", loss_orders_count)
print("Zero-profit orders:", zero_profit_orders)

There are 1,728 profitable orders and 272 loss-making orders. That means **13.6% of all orders are loss-making**.

The business still has a healthy overall positive profit, but the loss-making orders are worth investigating because they are concentrated in particular products and categories.

### Losses by product category

In [ ]:
loss_orders = df[df["Profit"] < 0].copy()

loss_by_category = (
    loss_orders
    .groupby("Product_Category")["Profit"]
    .agg(["count", "sum", "mean"])
    .sort_values("count", ascending=False)
)

loss_by_category

Office Supplies stands out immediately. It accounts for 224 of the 272 loss-making orders, or about **82.4%** of all loss-making orders.

That makes Office Supplies the first category I would investigate rather than assuming the losses are spread evenly across the business.

### Office Supplies: which products are driving the losses?

In [ ]:
office_supply_losses = (
    loss_orders[loss_orders["Product_Category"] == "Office Supplies"]
    .groupby("Product_Name")["Profit"]
    .agg(["count", "sum", "mean"])
    .sort_values("count", ascending=False)
)

office_supply_losses

### Are those products actually unprofitable overall?

In [ ]:
office_supply_profitability = (
    df[df["Product_Category"] == "Office Supplies"]
    .groupby("Product_Name")["Profit"]
    .agg(["count", "sum", "mean"])
    .sort_values("sum")
)

office_supply_profitability

This distinction is important.

A product can have many loss-making orders and still make a positive profit overall. Looking at all orders shows that **Paper Clips Box 500pc is genuinely unprofitable overall**, with total profit of -$79.95 across 45 orders.

Highlighters Neon Pack 6 is essentially break-even, with a slightly negative total profit of -$0.37.

### Loss order rate

In [ ]:
total_office_orders = (
    df[df["Product_Category"] == "Office Supplies"]
    .groupby("Product_Name")
    .size()
)

loss_office_orders = (
    loss_orders[loss_orders["Product_Category"] == "Office Supplies"]
    .groupby("Product_Name")
    .size()
)

loss_rate = (
    loss_office_orders
    .div(total_office_orders)
    .fillna(0)
    .mul(100)
    .sort_values(ascending=False)
)

loss_rate

Paper Clips Box 500pc has the highest loss order rate at **80%**. In other words, 36 of its 45 orders were loss-making.

Sticky Notes Multicolor 6-Pack and Highlighters Neon Pack 6 also have high loss rates.

This suggests that some low-value Office Supplies products may need a closer look at pricing, discounting, or shipping economics.

### Profitable vs loss-making orders

In [ ]:
profitability_comparison = (
    df.assign(
        Order_Status=np.select(
            [df["Profit"] < 0, df["Profit"] > 0],
            ["Loss", "Profit"],
            default="Zero Profit"
        )
    )
    .groupby("Order_Status")
    .agg(
        Orders=("Order_ID", "count"),
        Avg_Quantity=("Quantity", "mean"),
        Avg_Unit_Price=("Unit_Price", "mean"),
        Avg_Discount=("Discount_Percent", "mean"),
        Avg_Sales=("Total_Sales", "mean"),
        Avg_Shipping_Cost=("Shipping_Cost", "mean"),
        Avg_Profit=("Profit", "mean")
    )
)

profitability_comparison

The difference between profitable and loss-making orders is quite large.

Loss-making orders have much lower average unit prices and much lower average sales. Their average sales are only about $16.94 compared with $277.75 for profitable orders.

Interestingly, average shipping cost is not dramatically higher for loss-making orders. This suggests that the issue is less about unusually expensive shipping and more about shipping being a large cost relative to the value of small orders.

### Top 10 products by profit

In [ ]:
top_products_profit = (
    df.groupby("Product_Name")["Profit"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products_profit

Ergonomic Office Chair generated the highest total profit at **$15,104.58**, followed by Standing Desk Converter at **$14,694.35**.

This also shows why looking at both sales and profit matters: the product with the highest sales is not necessarily the product with the highest profit.

## 7. Geographic analysis

In [ ]:
region_analysis = (
    df.groupby("Region")
    .agg(
        Orders=("Order_ID", "count"),
        Sales=("Total_Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

region_analysis["Profit_Margin"] = (
    region_analysis["Profit"] / region_analysis["Sales"] * 100
)

region_analysis["AOV"] = (
    region_analysis["Sales"] / region_analysis["Orders"]
)

region_analysis.sort_values("Sales", ascending=False)

In [ ]:
region_analysis["Sales"].sort_values(ascending=False).plot(
    kind="bar", figsize=(9, 5)
)
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
region_analysis["Profit_Margin"].sort_values(ascending=False).plot(
    kind="bar", figsize=(9, 5)
)
plt.title("Profit Margin by Region")
plt.xlabel("Region")
plt.ylabel("Profit Margin (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

Europe generated the highest total sales at **$137,006.20**, while North America had the highest number of orders at 578 and the highest profit margin at about **33.80%**.

Europe's AOV was about **$272.38**, compared with about **$231.62** in North America. This helps explain why Europe generated slightly more sales despite having fewer orders.

### Country-level performance

In [ ]:
country_analysis = (
    df.groupby("Country")
    .agg(
        Orders=("Order_ID", "count"),
        Sales=("Total_Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

country_analysis["Profit_Margin"] = (
    country_analysis["Profit"] / country_analysis["Sales"] * 100
)

country_analysis["AOV"] = (
    country_analysis["Sales"] / country_analysis["Orders"]
)

country_analysis.sort_values("Sales", ascending=False)

In [ ]:
country_analysis.sort_values("Profit", ascending=False).head(5)

Mexico was the largest country by both sales and total profit, followed by Canada and the United States.

The same countries that lead sales also lead total profit, so there is no major mismatch between revenue leadership and profit leadership at the country level.

## 8. Customer segment analysis

In [ ]:
segment_analysis = (
    df.groupby("Customer_Segment")
    .agg(
        Orders=("Order_ID", "count"),
        Sales=("Total_Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

segment_analysis["AOV"] = (
    segment_analysis["Sales"] / segment_analysis["Orders"]
)

segment_analysis["Profit_Margin"] = (
    segment_analysis["Profit"] / segment_analysis["Sales"] * 100
)

segment_analysis.sort_values("Sales", ascending=False)

Consumer customers are the strongest segment in this dataset.

They generated:
- **1,006 orders**
- **$256,287.74 sales**
- **$87,300.36 profit**
- **$254.76 AOV**
- **34.06% profit margin**

Corporate customers are still an important source of revenue, but their profit margin is lower at about **30.44%**.

### Discounts by customer segment

In [ ]:
segment_discount = (
    df.groupby("Customer_Segment")["Discount_Percent"]
    .agg(["mean", "median"])
    .sort_values("mean", ascending=False)
)

segment_discount

Corporate customers receive noticeably higher discounts.

Their average discount is about **11.25%**, compared with **7.20%** for Consumer customers. The median shows the same pattern: 10% for Corporate versus 5% for Consumer.

This is a possible contributor to the lower Corporate profit margin, although the analysis does not by itself prove that discounting is the cause.

### Discount and profit relationship

In [ ]:
discount_profit_corr = df["Discount_Percent"].corr(df["Profit"])
print(f"Discount-Profit correlation: {discount_profit_corr:.3f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Discount_Percent"], df["Profit"], alpha=0.5)
plt.title("Discount vs Profit")
plt.xlabel("Discount (%)")
plt.ylabel("Profit ($)")
plt.tight_layout()
plt.show()

Correlation is useful here as a quick way to check whether discount and profit move together, but it should not be interpreted as proof that discounts cause changes in profit. Other factors such as product price, quantity, and shipping cost also affect profitability.

## 9. Payment method analysis

In [ ]:
payment_analysis = (
    df.groupby("Payment_Method")
    .agg(
        Orders=("Order_ID", "count"),
        Sales=("Total_Sales", "sum"),
        Profit=("Profit", "sum")
    )
)

payment_analysis["AOV"] = (
    payment_analysis["Sales"] / payment_analysis["Orders"]
)

payment_analysis["Profit_Margin"] = (
    payment_analysis["Profit"] / payment_analysis["Sales"] * 100
)

payment_analysis.sort_values("Sales", ascending=False)

Credit Card is the dominant payment method, with **797 orders** and **$192,717.89 in sales**.

Cash on Delivery has the highest AOV at about **$256.24** and the highest profit margin at about **33.33%**, although the differences in margin between payment methods are relatively small.

## 10. Key findings

### What I found

**1. Sales recovered in 2025.**  
Sales fell from $164,443.45 in 2023 to $155,150.73 in 2024, then recovered to $164,965.16 in 2025.

**2. Losses are concentrated in Office Supplies.**  
There were 272 loss-making orders, and 224 of them were Office Supplies orders. That is about 82.4% of all loss-making orders.

**3. Paper Clips Box 500pc is the clearest product problem.**  
It had 45 total orders, 36 of which were loss-making, giving it an 80% loss order rate and a total profit of -$79.95.

**4. Loss-making orders are generally low-value orders.**  
Average sales were about $16.94 for loss-making orders versus $277.75 for profitable orders. Shipping cost is not substantially higher on loss-making orders, so shipping appears to be a large cost relative to small order values.

**5. Europe leads total sales.**  
Europe generated $137,006.20 in sales, helped by an AOV of about $272.38.

**6. North America has the highest regional margin.**  
North America had a 33.80% profit margin, although the regional margins are fairly close overall.

**7. Consumers are the strongest customer segment.**  
Consumers generated the highest sales, profit, AOV, and profit margin.

**8. Corporate customers receive higher discounts.**  
Corporate customers averaged an 11.25% discount compared with 7.20% for Consumers, while also having the lowest segment profit margin.

## 11. Business recommendations

### 1. Review pricing for low-value Office Supplies

Paper Clips Box 500pc is the clearest example of a product that should be reviewed. The business could test a higher price, lower discount, or minimum order quantity.

### 2. Consider bundling small Office Supplies

Products with low order values can struggle to absorb shipping costs. Bundling related products could increase order value without necessarily requiring a large increase in acquisition or delivery costs.

### 3. Review Corporate discounting

Corporate customers generate substantial revenue, but their margin is lower than the other segments. Discount rules could be reviewed to see whether larger discounts are being given where they are not necessary.

### 4. Protect high-performing Furniture products

Several Furniture products appear among the strongest products by both sales and profit. These products could be given more attention in inventory planning and marketing.

### 5. Focus on high-value markets while watching efficiency

Europe is the strongest sales region, while North America has the highest margin. Both are useful markets, but the strategy should consider both revenue volume and profitability rather than focusing on sales alone.

## 12. Conclusion

Overall, the business is profitable, with total profit of **$158,872.32** and an overall profit margin of about **32.79%**.

The main opportunity is not simply to increase sales. A more useful priority is to reduce avoidable losses, especially in low-value Office Supplies orders, while continuing to support strong products and customer segments.

The biggest lesson from this analysis is that revenue alone does not tell the whole story. Looking at sales together with profit, order value, discounts, and loss rates gives a much clearer picture of how the business is performing.

## 13. Reproducibility

In [ ]:
# Dataset dimensions used in the analysis
print("Dataset shape:", df.shape)

# Final columns
print("\nColumns:")
print(df.columns.tolist())